# 10 - v0.1 Onboarding Journey

This notebook is the runnable version of the v0.1 onboarding probe. It uses only the kernel package surface and the native reasoning path:

1. Define a small schema with primary and secondary identity fields.
2. Write data with `sdk.ref`, `sdk.set`, and `sdk.add`.
3. Read with `sdk.get` and `sdk.run(Query(...))`.
4. Derive and accept a native candidate.
5. Export a kernel audit package.

No `agent`, `service`, `domains`, PyReason, ProbLog, or external API key is required.

## 0. Imports

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

# Works both when launched from repo root and from examples/.
cwd = Path.cwd().resolve()
repo_src = cwd / "src"
if not repo_src.exists() and cwd.name == "examples":
    repo_src = cwd.parent / "src"
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

In [ ]:
from kernel.adapters.souffle.package import ExportOptions
from kernel.sdk import Derivation, Entity, Field, Identity, Pred, Query, SDKStore, vars as sdk_vars

## 1. Define the schema

`User.user_id` is the primary identity. `User.locale` is a secondary identity coordinate with a default value. `name` is single-valued, `tag` is multi-valued, and `home` is an entity reference.

In [ ]:
class Country(Entity):
    code: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")


class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity(default="zh")
    name: str = Field(cardinality="single")
    tag: str = Field(cardinality="multi")
    home: Country = Field(cardinality="single")

## 2. Create refs and write facts

The v0.1 hardened path lets `sdk.set` and `sdk.add` materialize the entity identity and `<T>:exists` facts through the application write-plan path. The user does not need to start with `sdk.batch`.

In [ ]:
sdk = SDKStore([Country, User])

de = sdk.ref(Country, code="DE")
alice = sdk.ref(User, user_id="u1", locale="zh")

sdk.set(Country.name, de, "Germany")
sdk.set(User.name, alice, "Alice")
sdk.set(User.home, alice, de)
sdk.add(User.tag, alice, "admin")
sdk.add(User.tag, alice, "vip")

print(alice)

## 3. Read the entity snapshot

In [ ]:
snapshot = sdk.get(User, user_id="u1", locale="zh")

print(snapshot.name)
print(sorted(snapshot.tag))
print(snapshot.home)

## 4. Query with the SDK DSL

In [ ]:
with sdk_vars("u", "name") as (u, name):
    q = Query(
        head=[User(u), User.name(value=name)],
        where=[User(u), Pred("user:name", u, name)],
    )

rows = sdk.run(q)
rows

## 5. Derive and accept a native candidate

This native derivation turns an `admin` tag into a derived `audited` tag. It demonstrates that facts written through `sdk.set/add` are visible to the reasoning path.

In [ ]:
with sdk_vars("u", "loc", "derived") as (u, loc, derived):
    derivation = Derivation(
        id="journey.derived_tag",
        version="1.0.0",
        where=[User(u), u.locale == loc, Pred("user:tag", u, "admin"), derived == "audited"],
        head=User.tag(locale=loc, tag=derived),
    )

candidates = sdk.evaluate(derivation, mode="native")
print(len(candidates))

accepted = sdk.accept(candidates[0], approved_by="journey", note="accept derived audit tag")
accepted

In [ ]:
with sdk_vars("u") as (u,):
    derived_query = Query(
        head=User(u),
        where=[Pred("user:tag", u, "audited")],
    )

derived_rows = sdk.run(derived_query)
derived_rows

## 6. Export a kernel audit package

This verifies the kernel audit package boundary. It does not render a static site; rendered audit pages are a separate `service.static_ui` / monorepo delivery surface.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    out_dir = Path(tmp)
    sdk.export_package(out_dir, ExportOptions(package_kind="audit"))
    print((out_dir / "manifest.json").exists())
    print(sorted(p.name for p in out_dir.iterdir()))